# Pipeline Validation
Проверка корректности всех этапов подготовки данных перед LLM-обучением.

In [3]:
from pathlib import Path
import polars as pl

CATEGORY = "Pet_Supplies"
DATA_DIR = Path.cwd() / "prepared"
SEQ_DIR  = Path.cwd().parent / "sequences"

PATHS = {
    "items":    DATA_DIR / f"{CATEGORY}_items.parquet",
    "seqs":     DATA_DIR / f"{CATEGORY}_sequences.parquet",
    "cleaned":  DATA_DIR / f"{CATEGORY}_items_cleaned.parquet",
    "meta":     DATA_DIR / f"{CATEGORY}_items_with_metadata.parquet",
    "merged":   DATA_DIR / f"{CATEGORY}_items_merged.parquet",
}

ERRORS   = []
WARNINGS = []

def ok(msg):      print(f"  [OK]   {msg}")
def warn(msg):    WARNINGS.append(msg); print(f"  [WARN] {msg}")
def fail(msg):    ERRORS.append(msg);   print(f"  [FAIL] {msg}")
def section(msg): print(f"\n{'='*60}\n{msg}\n{'='*60}")

# Проверка существования файлов
section("Проверка наличия файлов")
for name, path in PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        ok(f"{name}: {path.name}  ({size_mb:.1f} MB)")
    else:
        fail(f"{name}: файл не найден — {path}")


Проверка наличия файлов
  [OK]   items: Pet_Supplies_items.parquet  (92.2 MB)
  [OK]   seqs: Pet_Supplies_sequences.parquet  (4.1 MB)
  [OK]   cleaned: Pet_Supplies_items_cleaned.parquet  (114.9 MB)
  [OK]   meta: Pet_Supplies_items_with_metadata.parquet  (115.6 MB)
  [OK]   merged: Pet_Supplies_items_merged.parquet  (93.0 MB)


In [4]:
# ============================================================
# Stage 1: prepape_data — items.parquet
# ============================================================
section("Stage 1: items.parquet (prepape_data выход)")

items = pl.read_parquet(PATHS["items"])
print(f"  Shape: {items.shape}")

# 1.1 Ожидаемый размер
if items.height >= 60_000:
    ok(f"Row count = {items.height:,}  (ожидаем ~63k после MIN_RATING_NUMBER=20)")
elif items.height >= 50_000:
    warn(f"Row count = {items.height:,}  (меньше ожидаемого ~63k)")
else:
    fail(f"Row count = {items.height:,}  (слишком мало)")

# 1.2 Обязательные колонки
required_cols = ["parent_asin", "main_category", "title", "rating_number",
                 "description_text", "features_text", "item_context"]
missing = [c for c in required_cols if c not in items.columns]
if missing:
    fail(f"Отсутствуют колонки: {missing}")
else:
    ok(f"Все обязательные колонки присутствуют")

# 1.3 Дубликаты по parent_asin
n_unique = items["parent_asin"].n_unique()
if n_unique == items.height:
    ok(f"parent_asin уникальны ({n_unique:,})")
else:
    fail(f"Дубликаты parent_asin: {items.height - n_unique:,} дублей")

# 1.4 Фильтр MIN_RATING_NUMBER=20 применён
min_rn = items["rating_number"].min()
if min_rn >= 20:
    ok(f"rating_number >= 20 для всех товаров (min={min_rn})")
else:
    fail(f"Есть товары с rating_number < 20 (min={min_rn}) — фильтр не применён")

# 1.5 Null в ключевых полях
for col in ["parent_asin", "title", "description_text"]:
    n_null = items[col].null_count()
    if n_null == 0:
        ok(f"{col}: null = 0")
    else:
        fail(f"{col}: {n_null:,} null значений")

# 1.6 Длина текстов (фильтры MIN_TITLE_LEN=50, MIN_DESC_LEN=500)
title_short = items.filter(pl.col("title").str.len_chars() <= 50).height
desc_short  = items.filter(pl.col("description_text").str.len_chars() <= 500).height
if title_short == 0:
    ok("Все title > 50 символов")
else:
    warn(f"{title_short:,} товаров с title <= 50 символов")
if desc_short == 0:
    ok("Все description > 500 символов")
else:
    warn(f"{desc_short:,} товаров с description <= 500 символов")

print("\n  Статистика rating_number:")
print(items["rating_number"].describe())


Stage 1: items.parquet (prepape_data выход)
  Shape: (63319, 11)
  [OK]   Row count = 63,319  (ожидаем ~63k после MIN_RATING_NUMBER=20)
  [OK]   Все обязательные колонки присутствуют
  [OK]   parent_asin уникальны (63,319)
  [OK]   rating_number >= 20 для всех товаров (min=20)
  [OK]   parent_asin: null = 0
  [OK]   title: null = 0
  [OK]   description_text: null = 0
  [OK]   Все title > 50 символов
  [OK]   Все description > 500 символов

  Статистика rating_number:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 63319.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 528.418168  │
│ std        ┆ 2806.542368 │
│ min        ┆ 20.0        │
│ 25%        ┆ 37.0        │
│ 50%        ┆ 79.0        │
│ 75%        ┆ 225.0       │
│ max        ┆ 132311.0    │
└────────────┴─────────────┘


In [5]:
# ============================================================
# Stage 1b: sequences.parquet
# ============================================================
section("Stage 1b: sequences.parquet (prepape_data выход)")

seqs = pl.read_parquet(PATHS["seqs"])
print(f"  Shape: {seqs.shape}")

# 2.1 Размер
if seqs.height >= 25_000:
    ok(f"Row count = {seqs.height:,}")
else:
    warn(f"Row count = {seqs.height:,}  (меньше ожидаемого ~27k+)")

# 2.2 Длины последовательностей
seq_len_stats = seqs["seq_len"].describe()
min_len = seqs["seq_len"].min()
max_len = seqs["seq_len"].max()
mean_len = seqs["seq_len"].mean()
if min_len >= 5:
    ok(f"seq_len >= 5 для всех пользователей (min={min_len}, mean={mean_len:.1f}, max={max_len})")
else:
    fail(f"Есть последовательности короче 5 (min={min_len})")

# 2.3 Дубликаты user_id
n_unique_users = seqs["user_id"].n_unique()
if n_unique_users == seqs.height:
    ok(f"user_id уникальны ({n_unique_users:,})")
else:
    fail(f"Дубликаты user_id: {seqs.height - n_unique_users:,} дублей")

# 2.4 Все товары в последовательностях есть в items
valid_asins = set(items["parent_asin"].to_list())
all_seq_asins = set(
    asin
    for seq in seqs["sequence"].to_list()
    for asin in (seq or [])
)
unknown_asins = all_seq_asins - valid_asins
if len(unknown_asins) == 0:
    ok(f"Все ASIN в последовательностях есть в items ({len(all_seq_asins):,} уникальных ASIN)")
else:
    fail(f"{len(unknown_asins):,} ASIN в последовательностях НЕ найдены в items")
    print(f"  Примеры: {list(unknown_asins)[:5]}")


Stage 1b: sequences.parquet (prepape_data выход)
  Shape: (101926, 3)
  [OK]   Row count = 101,926
  [OK]   seq_len >= 5 для всех пользователей (min=5, mean=7.6, max=100)
  [OK]   user_id уникальны (101,926)
  [OK]   Все ASIN в последовательностях есть в items (26,348 уникальных ASIN)


In [6]:
# ============================================================
# Stage 2: clean_data — items_cleaned.parquet
# ============================================================
section("Stage 2: items_cleaned.parquet (clean_data выход)")

cleaned = pl.read_parquet(PATHS["cleaned"])
print(f"  Shape: {cleaned.shape}")

# 3.1 Row count совпадает с items
if cleaned.height == items.height:
    ok(f"Row count = {cleaned.height:,}  (совпадает с items)")
else:
    fail(f"Row count = {cleaned.height:,}  (items={items.height:,}) — несовпадение")

# 3.2 Новые колонки присутствуют
for col in ["clean_title", "clean_description", "clean_features"]:
    if col in cleaned.columns:
        ok(f"Колонка {col} присутствует")
    else:
        fail(f"Колонка {col} отсутствует")

# 3.3 Null-rates для очищенных полей
n = cleaned.height
for col in ["clean_title", "clean_description", "clean_features"]:
    if col not in cleaned.columns:
        continue
    n_null = cleaned[col].null_count()
    pct = 100.0 * n_null / n
    if pct < 1.0:
        ok(f"{col}: null = {n_null:,} ({pct:.2f}%)")
    elif pct < 5.0:
        warn(f"{col}: null = {n_null:,} ({pct:.1f}%)  — возможны ошибки LLM")
    else:
        fail(f"{col}: null = {n_null:,} ({pct:.1f}%)  — слишком много ошибок")

# 3.4 clean_title не стал хуже оригинала (длина разумная)
if "clean_title" in cleaned.columns:
    empty_title = cleaned.filter(
        pl.col("clean_title").is_not_null() &
        (pl.col("clean_title").str.len_chars() < 5)
    ).height
    if empty_title == 0:
        ok("Все clean_title имеют длину >= 5 символов")
    else:
        warn(f"{empty_title:,} clean_title слишком короткие (< 5 символов)")

# 3.5 clean_features — тип List[str]
if "clean_features" in cleaned.columns:
    dtype = str(cleaned.schema["clean_features"])
    if "List" in dtype:
        ok(f"clean_features тип: {dtype}")
    else:
        fail(f"clean_features неожиданный тип: {dtype}  (ожидаем List[Utf8])")


Stage 2: items_cleaned.parquet (clean_data выход)
  Shape: (63319, 14)
  [OK]   Row count = 63,319  (совпадает с items)
  [OK]   Колонка clean_title присутствует
  [OK]   Колонка clean_description присутствует
  [OK]   Колонка clean_features присутствует
  [OK]   clean_title: null = 47 (0.07%)
  [OK]   clean_description: null = 47 (0.07%)
  [OK]   clean_features: null = 47 (0.07%)
  [OK]   Все clean_title имеют длину >= 5 символов
  [OK]   clean_features тип: List(String)


In [7]:
# ============================================================
# Stage 3: extract_data — items_with_metadata.parquet
# ============================================================
section("Stage 3: items_with_metadata.parquet (extract_data выход)")

meta = pl.read_parquet(PATHS["meta"])
print(f"  Shape: {meta.shape}")

# 4.1 Row count
if meta.height == items.height:
    ok(f"Row count = {meta.height:,}  (совпадает с items)")
else:
    fail(f"Row count = {meta.height:,}  (items={items.height:,}) — несовпадение")

# 4.2 Колонки структурированных полей
meta_cols = ["product_type", "pet_species", "life_stage", "primary_use_case", "material_or_active"]
for col in meta_cols:
    if col in meta.columns:
        ok(f"Колонка {col} присутствует")
    else:
        fail(f"Колонка {col} отсутствует")

# 4.3 Null-rates и покрытие
n = meta.height
for col in meta_cols:
    if col not in meta.columns:
        continue
    n_null = meta[col].null_count()
    pct_filled = 100.0 * (n - n_null) / n
    if pct_filled >= 70:
        ok(f"{col}: заполнено {pct_filled:.1f}%")
    elif pct_filled >= 40:
        warn(f"{col}: заполнено только {pct_filled:.1f}%")
    else:
        fail(f"{col}: заполнено только {pct_filled:.1f}%  — плохое покрытие")

# 4.4 material_or_active — тип List[str] или null (не строка!)
if "material_or_active" in meta.columns:
    dtype = str(meta.schema["material_or_active"])
    if "List" in dtype:
        ok(f"material_or_active тип: {dtype}")
    else:
        fail(f"material_or_active неожиданный тип: {dtype}  — ожидаем List[Utf8]")

# 4.5 Допустимые значения pet_species
VALID_SPECIES = {"dog", "cat", "fish", "bird", "small_animal", "reptile", "horse", "other"}
if "pet_species" in meta.columns:
    actual = set(meta["pet_species"].drop_nulls().unique().to_list())
    invalid = actual - VALID_SPECIES
    if not invalid:
        ok(f"pet_species: только допустимые значения ({sorted(actual)})")
    else:
        warn(f"pet_species: неожиданные значения: {invalid}")

# 4.6 Допустимые значения life_stage
VALID_STAGES = {"puppy", "kitten", "adult", "senior", "all"}
if "life_stage" in meta.columns:
    actual = set(meta["life_stage"].drop_nulls().unique().to_list())
    invalid = actual - VALID_STAGES
    if not invalid:
        ok(f"life_stage: только допустимые значения ({sorted(actual)})")
    else:
        warn(f"life_stage: неожиданные значения: {invalid}")


Stage 3: items_with_metadata.parquet (extract_data выход)
  Shape: (63319, 19)
  [OK]   Row count = 63,319  (совпадает с items)
  [OK]   Колонка product_type присутствует
  [OK]   Колонка pet_species присутствует
  [OK]   Колонка life_stage присутствует
  [OK]   Колонка primary_use_case присутствует
  [OK]   Колонка material_or_active присутствует
  [OK]   product_type: заполнено 99.8%
  [OK]   pet_species: заполнено 99.9%
  [OK]   life_stage: заполнено 85.9%
  [OK]   primary_use_case: заполнено 97.7%
  [OK]   material_or_active: заполнено 98.8%
  [OK]   material_or_active тип: List(String)
  [WARN] pet_species: неожиданные значения: {'ant', 'cat,dog,horse,small_animal', 'frog', 'ferret', 'dog,cat,small_animal', 'all', 'cat,dog', 'hermit_crab', 'pig', 'dog,cat,other', 'dog,cat,horse', 'chicken', 'rabbit', 'goat', 'poultry', 'turtle', 'insect', 'dog,cat'}
  [WARN] life_stage: неожиданные значения: {'small', 'newborn', 'foal', 'puppy,kitten', 'growing', 'senior,all', 'puppy,kitten,adult

In [8]:
# ============================================================
# Stage 4: enrich_data — items_merged.parquet (ГЛАВНАЯ ПРОВЕРКА)
# ============================================================
section("Stage 4: items_merged.parquet (enrich_data выход)")

merged = pl.read_parquet(PATHS["merged"])
print(f"  Shape: {merged.shape}")
print(f"  Columns: {merged.columns}")

# 5.1 Row count (inner join мог отфильтровать, но не должен)
if merged.height == items.height:
    ok(f"Row count = {merged.height:,}  (совпадает с исходными items — inner join ничего не потерял)")
elif merged.height >= items.height * 0.99:
    warn(f"Row count = {merged.height:,}  (потеряно {items.height - merged.height:,} строк при inner join)")
else:
    fail(f"Row count = {merged.height:,}  (потеряно {items.height - merged.height:,} строк — много!)")

# 5.2 Дубликаты parent_asin
n_unique = merged["parent_asin"].n_unique()
if n_unique == merged.height:
    ok(f"parent_asin уникальны ({n_unique:,})")
else:
    fail(f"Дубликаты parent_asin: {merged.height - n_unique:,}")

# 5.3 Рабочие поля title и description_text (cleaned)
n = merged.height
for col in ["title", "description_text", "features_text"]:
    if col not in merged.columns:
        fail(f"Колонка {col} отсутствует")
        continue
    n_null = merged[col].null_count()
    pct = 100.0 * n_null / n
    if pct < 0.1:
        ok(f"{col}: null = {n_null:,} ({pct:.2f}%)")
    elif pct < 2.0:
        warn(f"{col}: null = {n_null:,} ({pct:.1f}%)  — нет fallback на original")
    else:
        fail(f"{col}: null = {n_null:,} ({pct:.1f}%)")

# 5.4 item_context — не пустой, разумная длина
if "item_context" in merged.columns:
    ctx_len = merged["item_context"].str.len_chars()
    n_empty = (ctx_len == 0).sum()
    n_short = (ctx_len < 100).sum()
    if n_empty == 0 and n_short == 0:
        ok(f"item_context: min={ctx_len.min()}, median={ctx_len.median():.0f}, max={ctx_len.max()} символов")
    elif n_empty > 0:
        fail(f"item_context: {n_empty:,} пустых строк")
    else:
        warn(f"item_context: {n_short:,} очень коротких (<100 символов)")
else:
    fail("Колонка item_context отсутствует")

# 5.5 original_title сохранён (для fallback)
for col in ["original_title", "original_description"]:
    if col in merged.columns:
        ok(f"Оригинальные поля сохранены: {col}")
    else:
        warn(f"Оригинальное поле {col} отсутствует — fallback невозможен")

# 5.6 material_or_active — тип List[str]
if "material_or_active" in merged.columns:
    dtype = str(merged.schema["material_or_active"])
    if "List" in dtype:
        ok(f"material_or_active тип: {dtype}")
    else:
        fail(f"material_or_active тип: {dtype}  — ожидаем List")


Stage 4: items_merged.parquet (enrich_data выход)
  Shape: (63319, 19)
  Columns: ['parent_asin', 'title', 'description_text', 'original_title', 'original_description', 'main_category', 'categories_text', 'features_text', 'original_features_text', 'store', 'average_rating', 'rating_number', 'price', 'product_type', 'pet_species', 'life_stage', 'primary_use_case', 'material_or_active', 'item_context']
  [OK]   Row count = 63,319  (совпадает с исходными items — inner join ничего не потерял)
  [OK]   parent_asin уникальны (63,319)
  [OK]   title: null = 47 (0.07%)
  [OK]   description_text: null = 47 (0.07%)
  [WARN] features_text: null = 84 (0.1%)  — нет fallback на original
  [OK]   item_context: min=151, median=1245, max=4119 символов
  [OK]   Оригинальные поля сохранены: original_title
  [OK]   Оригинальные поля сохранены: original_description
  [OK]   material_or_active тип: List(String)


In [9]:
# ============================================================
# Cross-stage: согласованность merged с sequences
# ============================================================
section("Cross-stage: sequences vs merged items")

merged_asins = set(merged["parent_asin"].to_list())

# Все ASIN из sequences должны быть в merged
unknown_in_merged = all_seq_asins - merged_asins
if len(unknown_in_merged) == 0:
    ok(f"Все {len(all_seq_asins):,} ASIN из sequences найдены в merged")
else:
    fail(f"{len(unknown_in_merged):,} ASIN из sequences НЕ найдены в merged!")
    print(f"  Примеры: {list(unknown_in_merged)[:5]}")
    # Сколько пользователей пострадали
    affected_users = seqs.filter(
        pl.col("sequence").list.eval(
            pl.element().is_in(list(unknown_in_merged))
        ).list.any()
    ).height
    warn(f"  Затронуто пользователей: {affected_users:,} из {seqs.height:,}")

# Проверка что merged имеет более широкий или равный набор ASIN чем sequences
coverage = len(all_seq_asins & merged_asins) / len(all_seq_asins) * 100
if coverage >= 99.0:
    ok(f"Покрытие sequence ASIN в merged: {coverage:.2f}%")
elif coverage >= 95.0:
    warn(f"Покрытие sequence ASIN в merged: {coverage:.1f}%")
else:
    fail(f"Покрытие sequence ASIN в merged: {coverage:.1f}%  — критически мало")


Cross-stage: sequences vs merged items
  [OK]   Все 26,348 ASIN из sequences найдены в merged
  [OK]   Покрытие sequence ASIN в merged: 100.00%


In [10]:
# ============================================================
# Итоговый отчёт
# ============================================================
section("ИТОГОВЫЙ ОТЧЁТ")

print(f"  FAIL:  {len(ERRORS)}   критических ошибок")
print(f"  WARN:  {len(WARNINGS)} предупреждений")

if ERRORS:
    print("\n  Критические ошибки:")
    for e in ERRORS:
        print(f"    [FAIL] {e}")

if WARNINGS:
    print("\n  Предупреждения:")
    for w in WARNINGS:
        print(f"    [WARN] {w}")

if not ERRORS and not WARNINGS:
    print("\n  Все проверки пройдены! Данные готовы к следующему этапу.")
elif not ERRORS:
    print("\n  Критических ошибок нет. Предупреждения не блокируют пайплайн.")
else:
    print("\n  Есть критические ошибки — нужно исправить перед продолжением!")

# Сводная таблица размеров
print("\n  Сводка по размерам датафреймов:")
print(f"  {'Файл':<35} {'Строк':>10} {'Колонок':>8}")
print(f"  {'-'*55}")
for name, df_ in [("items", items), ("sequences", seqs),
                   ("cleaned", cleaned), ("meta", meta), ("merged", merged)]:
    print(f"  {name:<35} {df_.height:>10,} {len(df_.columns):>8}")


ИТОГОВЫЙ ОТЧЁТ
  FAIL:  0   критических ошибок
  WARN:  3 предупреждений

  Предупреждения:
    [WARN] pet_species: неожиданные значения: {'ant', 'cat,dog,horse,small_animal', 'frog', 'ferret', 'dog,cat,small_animal', 'all', 'cat,dog', 'hermit_crab', 'pig', 'dog,cat,other', 'dog,cat,horse', 'chicken', 'rabbit', 'goat', 'poultry', 'turtle', 'insect', 'dog,cat'}
    [WARN] life_stage: неожиданные значения: {'small', 'newborn', 'foal', 'puppy,kitten', 'growing', 'senior,all', 'puppy,kitten,adult,senior', 'juvenile', 'young', 'baby', 'juveniles', 'adult,senior', 'kitten,puppy'}
    [WARN] features_text: null = 84 (0.1%)  — нет fallback на original

  Критических ошибок нет. Предупреждения не блокируют пайплайн.

  Сводка по размерам датафреймов:
  Файл                                     Строк  Колонок
  -------------------------------------------------------
  items                                   63,319       11
  sequences                              101,926        3
  cleaned      

In [11]:

# ============================================================
# Post-processing: нормализация merged перед LLM-этапом
# ============================================================
section("Post-processing: нормализация merged")

# --- 1. Fallback для features_text ---
# Для 84 товаров clean_features оказался null — подставляем оригинал
if "original_features_text" in merged.columns:
    before = merged["features_text"].null_count()
    merged = merged.with_columns(
        pl.when(pl.col("features_text").is_null())
          .then(pl.col("original_features_text"))
          .otherwise(pl.col("features_text"))
          .alias("features_text")
    )
    after = merged["features_text"].null_count()
    if after < before:
        ok(f"features_text fallback применён: {before - after} строк восстановлено (осталось null: {after})")
    else:
        warn("features_text fallback не изменил данные")
else:
    warn("original_features_text отсутствует — fallback невозможен")

# --- 2. Нормализация pet_species ---
# Правила:
#   - 'all'          -> 'other'
#   - мультивидовые  -> берём первый вид из строки
#   - экзотика       -> 'other'
SPECIES_MAP = {
    "ant": "other", "frog": "other", "hermit_crab": "other", "pig": "other",
    "chicken": "other", "rabbit": "small_animal", "goat": "other",
    "poultry": "other", "turtle": "reptile", "insect": "other",
    "ferret": "small_animal", "all": "other",
}
VALID_SPECIES_SET = {"dog", "cat", "fish", "bird", "small_animal", "reptile", "horse", "other"}

def normalize_pet_species(val: str | None) -> str | None:
    if val is None:
        return None
    # Мультивидовые: берём первый вид
    if "," in val:
        val = val.split(",")[0].strip()
    val = SPECIES_MAP.get(val, val)
    return val if val in VALID_SPECIES_SET else "other"

normalized_species = merged["pet_species"].map_elements(
    normalize_pet_species, return_dtype=pl.Utf8
)
merged = merged.with_columns(normalized_species.alias("pet_species"))
remaining_invalid = set(merged["pet_species"].drop_nulls().unique().to_list()) - VALID_SPECIES_SET
if not remaining_invalid:
    ok(f"pet_species нормализован. Итоговые значения: {sorted(merged['pet_species'].drop_nulls().unique().to_list())}")
else:
    warn(f"pet_species: остались нестандартные значения: {remaining_invalid}")

# --- 3. Нормализация life_stage ---
STAGE_MAP = {
    "baby": "puppy",      "newborn": "puppy",   "foal": "puppy",
    "young": "puppy",     "growing": "puppy",   "juvenile": "puppy",
    "juveniles": "puppy", "kitten,puppy": "puppy", "puppy,kitten": "puppy",
    "adult,senior": "senior", "senior,all": "senior",
    "puppy,kitten,adult,senior": "all", "small": "puppy",
}
VALID_STAGES_SET = {"puppy", "kitten", "adult", "senior", "all"}

def normalize_life_stage(val: str | None) -> str | None:
    if val is None:
        return None
    if "," in val:
        val = STAGE_MAP.get(val, val.split(",")[0].strip())
    val = STAGE_MAP.get(val, val)
    return val if val in VALID_STAGES_SET else None

normalized_stage = merged["life_stage"].map_elements(
    normalize_life_stage, return_dtype=pl.Utf8
)
merged = merged.with_columns(normalized_stage.alias("life_stage"))
remaining_invalid = set(merged["life_stage"].drop_nulls().unique().to_list()) - VALID_STAGES_SET
if not remaining_invalid:
    ok(f"life_stage нормализован. Итоговые значения: {sorted(merged['life_stage'].drop_nulls().unique().to_list())}")
else:
    warn(f"life_stage: остались нестандартные значения: {remaining_invalid}")

# --- 4. Сохраняем обновлённый merged ---
merged.write_parquet(PATHS["merged"])
ok(f"merged сохранён: {PATHS['merged']}  ({merged.height:,} строк)")
print(f"\n  features_text null после fix: {merged['features_text'].null_count()}")
print(f"  pet_species null: {merged['pet_species'].null_count()}")
print(f"  life_stage  null: {merged['life_stage'].null_count()}")



Post-processing: нормализация merged
  [OK]   features_text fallback применён: 84 строк восстановлено (осталось null: 0)
  [OK]   pet_species нормализован. Итоговые значения: ['bird', 'cat', 'dog', 'fish', 'horse', 'other', 'reptile', 'small_animal']
  [OK]   life_stage нормализован. Итоговые значения: ['adult', 'all', 'kitten', 'puppy', 'senior']
  [OK]   merged сохранён: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/prepared/Pet_Supplies_items_merged.parquet  (63,319 строк)

  features_text null после fix: 0
  pet_species null: 81
  life_stage  null: 8924
